In [ ]:
from pathlib import Path
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from collections import defaultdict
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from huggingface_hub import hf_hub_download
import json, torch
from scipy.sparse import csr_matrix
from sklearn.metrics import f1_score, precision_score, recall_score, precision_recall_fscore_support, confusion_matrix
from google.colab import userdata
from huggingface_hub import HfApi
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
class SentimentAdapter(nn.Module):
    def __init__(self, hidden_dim=768, bottleneck_dim=64):
        super().__init__()
        self.down = nn.Linear(hidden_dim, bottleneck_dim, bias=False)
        self.act = nn.GELU()
        self.up = nn.Linear(bottleneck_dim, hidden_dim, bias=False)

        nn.init.zeros_(self.up.weight)

    def forward(self, x):
        return x + self.up(self.act(self.down(x)))

In [ ]:
def csr_to_torch_sparse(csr):
    coo = csr.tocoo()
    if coo.nnz == 0:
        indices = torch.empty((2,0), dtype=torch.long)
        values = torch.empty((0,), dtype=torch.float32)
    else:
        indices = torch.tensor(np.array([coo.row, coo.col]), dtype=torch.long)
        values = torch.tensor(coo.data, dtype=torch.float32)
    return torch.sparse_coo_tensor(indices, values, torch.Size(coo.shape))

def compute_metrics(y_true, y_pred, threshold=0.5):
    y_pred_bin = (y_pred > threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred_bin, average="macro", zero_division=0)
    return precision, recall, f1

In [ ]:
def weighted_focal_loss(inputs: torch.Tensor, targets: torch.Tensor, weights: torch.Tensor = None,
                        alpha: float = 0.25, gamma: float = 1.5, eps: float = 1e-8):
    p = torch.sigmoid(inputs)
    p = p.clamp(min=eps, max=1. - eps)
    # Compute the focal modulation for each element
    ce_loss = - (targets * torch.log(p) + (1 - targets) * torch.log(1 - p))
    pt = targets * p + (1 - targets) * (1 - p)  # p_t term
    focal_factor = (1 - pt) ** gamma
    # Apply alpha-balancing
    alpha_factor = targets * alpha + (1 - targets) * (1 - alpha)
    loss = alpha_factor * focal_factor * ce_loss
    # Apply class/sample weighting if provided
    if weights is not None:
        loss = loss * weights
    count_nonzero = torch.count_nonzero(weights)
    #return loss.mean()
    return loss.sum() / (count_nonzero + 1e-6)

In [ ]:
def apply_hierarchical_mask_to_sub_probs(p_top, p_sub, top_thresholds, top_to_sub_map):
    """  p_top: (N, T) numpy array of top probabilities
    p_sub: (N, S) numpy array of sub probabilities
    top_thresholds: array-like length T
    top_to_sub_map: one of:
        - dict: {top_idx: [sub_idx, ...]}
        - torch.sparse_coo_tensor or torch.Tensor (dense)
        - scipy.sparse matrix
        - numpy.ndarray (dense)
    Returns: p_sub masked (numpy array)  """

    if top_to_sub_map is None or top_thresholds is None:
        return p_sub

    top_thresholds = np.array(top_thresholds)
    top_pred_bin = (p_top > top_thresholds[None, :]).astype(np.int32)  # (N, T)

    # Sanity: ensure shapes align
    T = top_pred_bin.shape[1]
    S = p_sub.shape[1]
    if dense_map.shape != (T, S):
        # try to transpose if user stored (S, T)
        if dense_map.shape == (S, T):
            dense_map = dense_map.T
        else:
            raise ValueError(f"top_to_sub_map shape mismatch: expected ({T},{S}), got {dense_map.shape}")

    mask_counts = top_pred_bin.dot(dense_map)  # int counts
    mask = (mask_counts > 0).astype(float)     # 1.0 for allowed subs, 0.0 otherwise

    # If no top predicted for an example, choose fallback: allow all subs (original behavior)
    # (original code set mask[i,:]=1.0 if relevant_subs empty)
    no_top = (top_pred_bin.sum(axis=1) == 0)
    if no_top.any():
        mask[no_top, :] = 1.0

    return p_sub * mask

In [ ]:
class JointDataset(Dataset):
    def __init__(self, data, tokenizer, top_dim, sub_dim, max_len=256, label_map=None):
        self.data = data
        self.tokenizer = tokenizer
        self.top_dim = top_dim
        self.sub_dim = sub_dim
        self.max_len = max_len
        self.label_map = label_map

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item["text"]

        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        enc = {k: v.squeeze(0) for k, v in enc.items()}

        top = torch.tensor(item["top_cluster_ids"], dtype=torch.float32)
        if top.ndim == 0:  # single int label
            top = F.one_hot(top.long(), num_classes=self.top_dim).float()

        sub_ids = item.get("sub_cluster_ids", [])

        if isinstance(sub_ids, (list, np.ndarray, torch.Tensor)):
            if len(sub_ids) == self.sub_dim:
                arr = np.array(sub_ids, dtype=np.float32)
                indices = np.where(arr > 0)[0].tolist()
            else:
                # Otherwise, treat as list of indices
                indices = [int(i) for i in sub_ids if int(i) >= 0]
        else:
            # Fallback empty
            indices = []

        # Build sparse binary tensor
        if len(indices) == 0:
            sub_sparse = torch.sparse_coo_tensor(
                torch.empty((2, 0), dtype=torch.long),
                torch.empty((0,), dtype=torch.float32),
                (1, self.sub_dim)
            )
        else:
            idx_tensor = torch.tensor([[0] * len(indices), indices], dtype=torch.long)
            val_tensor = torch.ones(len(indices), dtype=torch.float32)
            sub_sparse = torch.sparse_coo_tensor(idx_tensor, val_tensor, (1, self.sub_dim))

        raw_sentiments = item.get("sentiments", {})
        sentiments = {}
        for k, v in raw_sentiments.items():
            mapped_v = v
            if self.label_map:
                mapped_v = self.label_map.get(v)

            if mapped_v is not None: # Only include if a valid (non-None) value is found/mapped
                sentiments[int(k)] = mapped_v

        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "top_labels": top,
            "sub_labels": sub_sparse,
            "sentiments": sentiments,
        }

    @staticmethod
    def collate_fn(batch):
        return {
            "input_ids": torch.stack([b["input_ids"] for b in batch]),
            "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
            "top_labels": torch.stack([b["top_labels"] for b in batch]),
            "sub_labels": [b["sub_labels"] for b in batch],  # keep sparse list
            "sentiments": [b["sentiments"] for b in batch],
        }

In [ ]:
class AspectAttention(nn.Module):
    def __init__(self, hidden_dim, aspect_emb_dim, d_k=256, d_v=256):
        super().__init__()
        self.Wq = nn.Linear(aspect_emb_dim, d_k, bias=False)
        self.Wk = nn.Linear(hidden_dim, d_k, bias=False)
        self.Wv = nn.Linear(hidden_dim, d_v, bias=False)
        self.Wo = nn.Linear(d_v, hidden_dim)

    def forward(self, enc, aspect_emb, attention_mask):
        """
        enc: (B, L, H)
        aspect_emb: (B, N, E)  <-- Now expects 3D input (Standard)
        """
        # Linear layers handle the (B, N) dimensions automatically
        Q = self.Wq(aspect_emb)          # (B, N, d_k)
        K = self.Wk(enc)                 # (B, L, d_k)
        V = self.Wv(enc)                 # (B, L, d_v)

        # Compute Scores: (B, N, d_k) x (B, d_k, L) -> (B, N, L)
        scores = torch.matmul(Q, K.transpose(1, 2)) / (Q.size(-1) ** 0.5)

        # Masking
        mask = attention_mask.squeeze(-1).unsqueeze(1) # (B, 1, L)
        scores = scores.masked_fill(mask == 0, -1e4) # Safe -1e4
        attn = F.softmax(scores, dim=-1)
        context = torch.matmul(attn, V)  # (B, N, d_v)

        return self.Wo(context)   #(B, N, H)

In [ ]:
class HierarchicalClassifier(nn.Module):
    def __init__(self, encoder_name, top_dim, sub_dim, dropout=0.2, subfolder=None): # Add subfolder
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name, subfolder=subfolder) # Pass subfolder
        h_dim = self.encoder.config.hidden_size
        self.top_dim = top_dim
        self.sub_dim = sub_dim
        self.aspect_emb_dim = h_dim
        self.head_aspect_attention = AspectAttention(
            hidden_dim=h_dim, aspect_emb_dim=self.aspect_emb_dim)
        self.head_query = nn.Parameter(torch.empty(1, top_dim, h_dim))
        nn.init.normal_(self.head_query, mean=0, std=0.02)
        self.head_gate_proj = nn.Linear(h_dim * 2, h_dim)
        # Stage 1 (top-level)
        self.head_norm = nn.LayerNorm(h_dim)
        self.head_weight = nn.Parameter(torch.randn(top_dim, h_dim))
        self.head_bias = nn.Parameter(torch.zeros(top_dim))

        # Initialization
        torch.nn.init.normal_(self.head_weight, std=0.02)
        self.head_dropout = nn.Dropout(dropout)

        # Stage 2 (sub-level)

        self.sub_aspect_attention = AspectAttention(
            hidden_dim=h_dim, aspect_emb_dim=self.aspect_emb_dim)
        self.sub_query = nn.Parameter(torch.empty(1, sub_dim, h_dim))
        nn.init.normal_(self.sub_query, mean=0, std=0.02)
        self.sub_gate_proj = nn.Linear(h_dim * 2, h_dim)

        self.sub_norm = nn.LayerNorm(h_dim)
        self.sub_weight = nn.Parameter(torch.randn(sub_dim, h_dim))
        self.sub_bias = nn.Parameter(torch.zeros(sub_dim))

        # Initialization
        torch.nn.init.normal_(self.sub_weight, std=0.02)
        self.sub_dropout = nn.Dropout(dropout)

        # Mapping-related attributes
        self.top_to_sub_map = None          # sparse tensor (top_dim × sub_dim)
        self.top_to_sub_dense = None        # dense cache for matmul
        self.top_dim = top_dim
        self.sub_dim = sub_dim
        self.top_pos_weight = None
        self.sub_pos_weight = None
        self.sub_prob_weight = nn.Parameter(torch.full((1,), 0.4))

    def set_top_to_sub_map(self, mapping: dict):
        self.top_to_sub_dict = mapping  # keep for loss computation

        rows, cols = [], []
        for t, subs in mapping.items():
            rows.extend([t] * len(subs))
            cols.extend(subs)

        indices = torch.tensor([rows, cols], dtype=torch.long)
        values = torch.ones(len(rows), dtype=torch.float32)
        sparse_map = torch.sparse_coo_tensor(indices, values, (self.top_dim, self.sub_dim))
        self.top_to_sub_map = sparse_map.coalesce()
        self.top_to_sub_dense = None


    def _get_dense_map(self, device, dtype):
        """
        Safely gets (or builds) dense mapping tensor.
        - Auto moves to correct device/dtype (handles AMP).
        - Caches result for reuse.
        """
        if self.top_to_sub_dense is None or self.top_to_sub_dense.device != device:
            dense_map = self.top_to_sub_map.to_dense().to(device)
            self.top_to_sub_dense = dense_map
        # ensure dtype matches current autocast precision
        if self.top_to_sub_dense.dtype != dtype:
            self.top_to_sub_dense = self.top_to_sub_dense.to(dtype)
        return self.top_to_sub_dense

    def forward(self, input_ids, attention_mask, gating_mode="add", T=1, t=1):
        # Encoder
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True, return_dict=True)
        enc = outputs.last_hidden_state
        batch_size = enc.size(0)
        mask = attention_mask.unsqueeze(-1).float() #(B, L, 1)
        pooled = (enc * mask).sum(1) / mask.sum(1).clamp(min=1.0)
        top_query_expanded = self.head_query.expand(batch_size, -1, -1) # (B, N, H)
        aspect_context = self.head_aspect_attention(enc, top_query_expanded, mask) # (B, N, H)
        pooled_expanded = pooled.unsqueeze(1).expand(-1, self.top_dim, -1) # (B, N, H)

        # Concatenate: (B, N, 2*H)
        combined = torch.cat([pooled_expanded, aspect_context], dim=2)
        # Compute Gate Alpha: (B, N, H)
        alpha = torch.sigmoid(self.head_gate_proj(combined))

        # Fuse: Alpha controls how much we use Pooled vs Attention
        fused_embedding = alpha * pooled_expanded + (1 - alpha) * aspect_context
        fused_norm = self.head_norm(self.head_dropout(fused_embedding))

        # Top-Level Prediction
        top_logits = (fused_norm * self.head_weight).sum(dim=-1) + self.head_bias
        # Now opened for sub
        p_top = torch.sigmoid(top_logits)

        # Compute sub-cluster prior
        batch_size, device, dtype = p_top.size(0), p_top.device, p_top.dtype
        if self.top_to_sub_map is not None:
            map_dense = self._get_dense_map(device, dtype)
            sub_prior = torch.matmul(p_top, map_dense)
            sub_prior.clamp_(0.0, 1.0)
        else:
            sub_prior = torch.zeros(batch_size, self.sub_dim, device=device, dtype=dtype)

        # Move thresholds to the same device as p_top
        hard_mask = (p_top >= self.top_thresholds.to(p_top.device)).float()
        sub_query_expanded = self.sub_query.expand(batch_size, -1, -1) # (B, N, H)
        sub_aspect_context = self.sub_aspect_attention(enc, sub_query_expanded, mask) # (B, N, H)
        # Gating Logic
        # Concatenate: (B, N, 2*H)
        pooled_expanded = pooled.unsqueeze(1).expand(-1, self.sub_dim, -1)
        combined = torch.cat([pooled_expanded, sub_aspect_context], dim=2)
        # Compute Gate Alpha: (B, N, H)
        alpha = torch.sigmoid(self.sub_gate_proj(combined))

        # Fuse: Alpha controls how much we use Pooled vs Attention
        fused_embedding = alpha * pooled_expanded + (1 - alpha) * sub_aspect_context
        fused_norm = self.sub_norm(self.sub_dropout(fused_embedding))

        sub_logits = (fused_norm * self.sub_weight).sum(dim=-1) + self.sub_bias

        # Hierarchical gating
        if gating_mode == "add":
            sub_logits = sub_logits + 0.5 * sub_prior.detach()
        elif gating_mode == "mul":
            sub_logits = sub_logits * (1.0 + 0.5 * sub_prior.detach())
        x = enc.clone()
        for adapter in self.sentiment_adapters:
            x = adapter(x)
        pooled_sent = (x * mask).sum(1) / mask.sum(1).clamp(min=1.0)
        return top_logits, sub_logits, pooled_sent, x, enc, pooled
    

    def compute_loss(self, top_logits, y_top, sub_logits, y_sub_sparse_list):
        device = sub_logits.device
        y_top = y_top.to(dtype=torch.float32, device=device)
        top_loss = F.binary_cross_entropy_with_logits(top_logits, y_top, pos_weight=self.top_pos_weight.to(device) if self.top_pos_weight is not None else None)

        y_sub_dense = torch.stack([s.to_dense().squeeze(0).to(device) for s in y_sub_sparse_list])

        device, dtype = y_top.device, y_top.dtype
        if self.top_to_sub_map is not None:
            map_dense = self._get_dense_map(device, dtype)

        with torch.no_grad():
            mask = torch.zeros_like(y_sub_dense)
            valid_sub_mask = torch.matmul(y_top, map_dense)
            mask = (valid_sub_mask > 0).float()
            no_top_prediction = (mask.sum(dim=1) == 0)
            if no_top_prediction.any():
                mask[no_top_prediction] = 1.0

            if mask.sum() == 0: mask = torch.ones_like(mask)

        if self.sub_pos_weight is not None:
            pos_w = self.sub_pos_weight.to(device).unsqueeze(0).repeat(y_sub_dense.size(0),1)
            sub_weights = pos_w * mask
        else:
            sub_weights = mask

        sub_loss = weighted_focal_loss(sub_logits, y_sub_dense, weights=sub_weights)
        return top_loss, sub_loss

In [ ]:
class JointModel(HierarchicalClassifier):
    def __init__(self, encoder_name, top_dim, sub_dim, dropout=0.2, subfolder=None): # Add subfolder
        super().__init__(encoder_name, top_dim, sub_dim, dropout=dropout, subfolder=subfolder) # Pass subfolder
        self.sentiment_adapters = nn.ModuleList([
            SentimentAdapter(768, 64) for _ in range(5)])
        h_dim = self.encoder.config.hidden_size
        self.aspect_emb_dim = h_dim
        self.sent_aspect_attention = AspectAttention(
            hidden_dim=h_dim, aspect_emb_dim=self.aspect_emb_dim, d_k=h_dim, d_v=h_dim)
        self.sent_gate_proj = nn.Linear(h_dim * 2, h_dim * 2)
        sent_in = 2*h_dim+self.aspect_emb_dim
        self.sent_norm1 = nn.LayerNorm(h_dim)
        self.sent_norm2 = nn.LayerNorm(h_dim)
        self.sent_norm3 = nn.LayerNorm(h_dim)

        self.sent_proj = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(sent_in, sent_in//2),
            nn.GELU(),
            nn.Linear(sent_in//2, sent_in//4),
            nn.GELU(),
            nn.Linear(sent_in//4, 2))

    def predict_sentiments_from_batch(self, encodings, sentiments_dict_batch, attention_mask, pooled_enc):
        device = encodings.device
        sent_logits_list = []
        sent_labels_list = []
        sent_example_map = []
        sent_sub = []
        B = encodings.size(0)

        for i in range(B):
            sd = sentiments_dict_batch[i] if sentiments_dict_batch is not None else {}
            if not sd: continue

            sub_ids = list(sd.keys())
            labels = list(sd.values())

            # 1. Get Embeddings for ACTIVE aspects only (Targeted)
            sub_ids_tensor = torch.tensor(sub_ids, dtype=torch.long, device=device)
            aspect_embs = self.sub_query[:, sub_ids_tensor,:] # (1, N_active, E)

            # 2. Get Encoder Output for this batch item
            curr_enc = encodings[i].unsqueeze(0)        # (1, L, H)
            curr_mask = attention_mask[i].unsqueeze(0)  # (1, L)

            # 3. Apply Attention (Targeted)
            # Returns (1, N_active, H)
            aspect_context = self.sent_aspect_attention(curr_enc, aspect_embs, curr_mask).squeeze(0)
            pooled = pooled_enc[i].unsqueeze(0)      # (1, H)
            pooled_expanded = pooled.expand(sub_ids_tensor.size(0), -1)

            # 4. Concatenate Context + Aspect Identity (The "Feature Construction" Fix)
            # (1, N_active, H+E)
            combined = torch.cat([pooled_expanded, aspect_context], dim=-1)

            # 5. Predict
            combined = combined.float()
            alpha = torch.sigmoid(self.sent_gate_proj(combined))
            alpha_pooled, alpha_context = alpha.chunk(2, dim=-1)

            aspect_emb = aspect_embs.squeeze(0)
            pooled_expanded = self.sent_norm1(pooled_expanded)
            aspect_context = self.sent_norm2(aspect_context)
            fused_embedding = torch.cat([alpha_pooled * pooled_expanded, alpha_context * aspect_context], dim=-1)

            feature = torch.cat([aspect_emb, fused_embedding], dim=-1)

            sent_logits = self.sent_proj(feature).squeeze(-1)

            sent_logits_list.append(sent_logits)
            sent_labels_list.append(torch.tensor(labels, dtype=torch.float32, device=device))
            sent_example_map.append(i)
            sent_sub.append(sub_ids)

        return sent_logits_list, sent_labels_list, sent_example_map, sent_sub

    def forward(self, input_ids, attention_mask, sentiments_dict_batch=None, gating_mode="add"):
        top_logits, sub_logits, pooled_sent, enc_sent, enc, pooled = super().forward(input_ids, attention_mask, gating_mode=gating_mode)

        if sentiments_dict_batch is not None:
            sent_logits_adap, sent_labels_list, sent_example_map, sent_sub = self.predict_sentiments_from_batch(
                enc_sent, sentiments_dict_batch, attention_mask, pooled_sent)
            sent_logits_list, sent_labels_list, sent_example_map, sent_sub = self.predict_sentiments_from_batch(
                enc, sentiments_dict_batch, attention_mask, pooled)
        return {
            "top_logits": top_logits,
            "sub_logits": sub_logits,
            "sent_logits_list": sent_logits_list,
            "sent_labels_list": sent_labels_list,
            "sent_logits_adap": sent_logits_adap,
            "sent_example_map": sent_example_map,
            "sent_sub": sent_sub
        }


    def compute_joint_loss(self, outputs, y_top, y_sub_sparse_list):
        """
        Compute combined loss:
           total = (alpha * top_loss + beta * sub_loss + hier_reg_weight * hier_reg) + gamma * sentiment_loss
        Returns:
           total_loss (tensor),
           metrics tuple: (top_loss_float, sub_loss_float, sent_loss_float, hier_reg_float)
        Notes:
           - sentiment loss is averaged across all annotated aspects in the batch
           - if there are no sentiment annotations in batch, sentiment loss = 0
        """
        device = outputs["top_logits"].device
        lambda_margin = 0.1
        margin = 0.5
        # hierarchical loss computed by base
        top_loss, sub_loss= self.compute_loss(
            outputs["top_logits"], y_top, outputs["sub_logits"], y_sub_sparse_list)

        all_logits = outputs["sent_logits_list"]
        all_labels = outputs["sent_labels_list"]
        sent_loss = 0
        if len(all_logits) > 0:
            all_logits = torch.cat(all_logits, dim=0)   # (N_total,)
            all_labels = torch.cat(all_labels, dim=0)   # (N_total,)
            ce_loss = F.cross_entropy(all_logits, all_labels.long())

            diff = torch.abs(all_logits[:, 1] - all_logits[:, 0])
            margin_loss = torch.relu(margin - diff).mean()
            sent_loss = ce_loss + lambda_margin * margin_loss

        else:
            sent_loss = torch.tensor(0.0, device=device)
        return top_loss, sub_loss, sent_loss

In [ ]:
def load_tokenizer_and_model_from_hf(cfg, top_dim, sub_dim, device):
    hf_repo_id = cfg.get("hf_repo")
    hf_subfolder = cfg.get("hf_encoder_subfolder")

    checkpoint_filename = "final_stage2_v6/best_stage2.pt"
    print(f"🔹 Loading Tokenizer & Base Architecture from: {hf_repo_id} (subfolder: {hf_subfolder})")

    try:
        tokenizer = AutoTokenizer.from_pretrained(hf_repo_id, subfolder=hf_subfolder)
        print(f" Tokenizer loaded.")
    except Exception as e:
        print(f" Failed to load tokenizer: {e}")
        tokenizer = AutoTokenizer.from_pretrained(hf_repo_id)

    print(f" Initializing Base Architecture from: {hf_repo_id} (subfolder: {hf_subfolder})")

    # Initialize JointModel with YOUR domain encoder as the base
    model = JointModel(
            hf_repo_id,  # Pass repo_id
            subfolder=hf_subfolder, # Pass subfolder to JointModel
            top_dim=top_dim,
            sub_dim=sub_dim,
            dropout=0.2).to(device)

    print(f"⬇️ Downloading weights from: {hf_repo_id}/{checkpoint_filename} ...")
    try:
        cached_path = hf_hub_download(
            repo_id=hf_repo_id,
            filename=checkpoint_filename)

        checkpoint = torch.load(cached_path, map_location=device)

        if "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        else:
            state_dict = checkpoint # Fallback if it was saved as direct state_dict

        missing, unexpected = model.load_state_dict(state_dict, strict=False)

        print(f" Trained weights loaded successfully!")
        if missing: print(f"   Missing keys: {len(missing)} (Ensure this matches expectation)")
        if unexpected: print(f"   Unexpected keys: {len(unexpected)}", unexpected)

    except Exception as e:
        print(f" Critical Error: Could not load checkpoint from Hugging Face.")

    return tokenizer, model

In [ ]:
def make_optimizer_and_scheduler(model, cfg, phase, steps_per_epoch):
    """
    Create optimizer and scheduler.
    - phase: "freeze" or "unfreeze" to decide which params are trainable.
    """
    # Ensure requires_grad flags already set by caller (freeze/unfreeze)
    named = list(model.named_parameters())

    sent_params = []
    heads_params = []
    encoder_params = []

    for n, p in named:
        if not p.requires_grad:
            continue
        lname = n.lower()
        if("sent" in lname):
            sent_params.append(p)
        elif("head" in lname) or ("sub" in lname):
            heads_params.append(p)
        elif("encoder" in lname and "sent" not in lname):
            encoder_params.append(p)

    param_groups = []
    if encoder_params:
        param_groups.append({"params": encoder_params, "lr": cfg.get("lr_encoder", 1e-5), "weight_decay": cfg.get("weight_decay", 0.1)})
    if sent_params:
        param_groups.append({"params": sent_params, "lr": cfg.get("lr_sent", 1e-4), "weight_decay": cfg.get("weight_decay", 0.1)})
    if heads_params:
        param_groups.append({"params": heads_params, "lr": cfg.get("lr_heads", 2e-5), "weight_decay": cfg.get("weight_decay", 0.1)})

    optimizer = torch.optim.AdamW(param_groups, betas=(0.9, 0.999), eps=1e-8)

    # scheduler: linear warmup + decay
    if phase == "freeze":
        total_steps = steps_per_epoch*cfg.get("freeze_epochs")
        warmup_ratio = 0.1
    elif phase == "unfreeze":
        total_steps = steps_per_epoch*(cfg.get("epochs")+1-cfg.get("freeze_epochs"))
        warmup_ratio = 0.1

    num_warmup_steps = max(1, int(total_steps * warmup_ratio))
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps, total_steps)
    return optimizer, scheduler


In [ ]:
def class_wise_eval(y_true, y_pred_probs):
    n_classes = y_true.shape[1]
    print("-" * 85)
    print(f"{'Class ID':<10} | {'Threshold':<10} | {'F1':<10} | {'Precision':<10} | {'Recall':<10} | {'Support':<10}")
    print("-" * 85)
    final_avg_thresholds = [0.5]*n_classes
    class_metrics = {}
    macro_f1_scores = []

    for c in range(n_classes):
        # Apply the optimized threshold to the full dataset
        preds_bin = (y_pred_probs[:, c] > final_avg_thresholds[c]).astype(int)
        true_bin = y_true[:, c]

        # Calculate metrics
        precision = precision_score(true_bin, preds_bin, zero_division=0)
        recall = recall_score(true_bin, preds_bin, zero_division=0)
        f1 = f1_score(true_bin, preds_bin, zero_division=0)
        support = true_bin.sum() * 9

        macro_f1_scores.append(f1)

        # Store
        class_metrics[c] = {
            "threshold": final_avg_thresholds[c],
            "f1": f1,
            "precision": precision,
            "recall": recall,
            "support": support
        }

        # Print Row
        print(f"{c} | {final_avg_thresholds[c]:.3f}      | {f1:.4f}     | {precision:.4f}     | {recall:.4f}     | {int(support)}")

    print("-" * 85)
    avg_f1 = np.mean(macro_f1_scores)
    print(f" Final Macro F1 Score: {avg_f1:.4f}")

    return final_avg_thresholds, avg_f1, class_metrics

In [ ]:
def train_sentiment_head(cfg):
    DEVICE = torch.device(cfg["device"])
    sample_json = json.load(open(cfg["hierarchical_json"], "r", encoding="utf-8"))

    top_to_sub_map = defaultdict(set)
    for it in sample_json:
        for k, v in it.get("top_to_sub_ids", {}).items():
            top_to_sub_map[int(k)].update(v)
    top_to_sub_map = {k: list(v) for k, v in top_to_sub_map.items()}

    first = sample_json[0]
    top_dim = len(first["top_cluster_ids"])
    sub_dim = len(first["sub_cluster_ids"]) if "sub_cluster_ids" in first else 29 # Fallback if not directly available

    print("\n🔹 Loading tokenizer and pretrained Stage-2 weights from HF...")
    tokenizer, model, top_thresholds, sub_thresholds = load_tokenizer_and_model_from_hf(
        cfg=cfg,
        top_dim=top_dim,
        sub_dim=sub_dim,
        device=DEVICE)

    model.set_top_to_sub_map(top_to_sub_map)

    dataset = JointDataset(sample_json, tokenizer, top_dim=top_dim, sub_dim=sub_dim, max_len=cfg["max_len"], label_map=cfg["label_map"])
    valid_indices = np.load("/content/valid_indices (2).npy")
    all_indices = np.arange(len(dataset))
    train_indices = np.setdiff1d(all_indices, valid_indices)
    train_ds = torch.utils.data.Subset(dataset, train_indices)
    val_ds = torch.utils.data.Subset(dataset, valid_indices)

    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, collate_fn=JointDataset.collate_fn)
    val_loader = DataLoader(val_ds, batch_size=cfg["batch_size"], shuffle=False, collate_fn=JointDataset.collate_fn)

    eval_data = []
    for i in valid_indices:
      eval_data.append(sample_json[i].get("text"))

    freeze_epochs = cfg["freeze_epochs"]
    steps_per_epoch = len(train_loader)
    scaler = torch.amp.GradScaler('cuda', init_scale=2**8, enabled=cfg.get("use_amp"))

    start_epoch = 1
    current_step = 0 # Renamed from 'step' to 'current_step' for clarity in scope

    optimizer = None
    scheduler = None
    current_optimizer_phase = None # To track if the optimizer/scheduler needs re-creation

    for name, p in model.named_parameters():
        p.requires_grad = False # Default freeze
        if any(k in name.lower() for k in ["sent"]):
            p.requires_grad = True # Train New Head

    for epoch in range(start_epoch, cfg["epochs"] + 1):
        # Determine the target phase for the current epoch
        target_phase = "unfreeze"
        if cfg["freeze"] and epoch <= freeze_epochs:
            target_phase = "freeze"
            print(f"\n Epoch {epoch}: Encoder ,top head and sub head frozen → training sentiment head only.")
        else:
            for n, p in model.named_parameters():
                lname = n.lower()
                # Unfreeze LoRA, Heads, and Sentiment Projections
                if any(k in lname for k in ["encoder", "sub", "head", "sent"]):
                    p.requires_grad = True
                # Re-initialize optimizer and scheduler only if the phase has changed
        if target_phase != current_optimizer_phase:
            print(f"🔄 Phase change detected: {current_optimizer_phase} -> {target_phase}. Re-initializing optimizer/scheduler.")
            optimizer, scheduler = make_optimizer_and_scheduler(model, cfg, target_phase, epoch, steps_per_epoch)
            current_optimizer_phase = target_phase

        # Training
        model.train()
        top_loss = 0.0
        sub_loss = 0.0
        sent_loss = 0.0

        for batch in tqdm(train_loader, desc=f"Train E{epoch}"):
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            y_top = batch["top_labels"].to(DEVICE)
            y_sub = batch["sub_labels"].to(DEVICE)
            sentiments = batch["sentiments"]

            optimizer.zero_grad()
            with torch.amp.autocast(device_type="cuda", enabled=(cfg["use_amp"])):
                outputs = model(ids, mask, sentiments)
                top_l, sub_l, sent_l = model.compute_joint_loss(
                    outputs, y_top, y_sub, alpha=1.0, beta=1.0, gamma=1.0)
                if not torch.isfinite(top_l+sent_l+sub_l):
                    print("NON-FINITE LOSS detected at step", current_step)
                    optimizer.zero_grad()
                    continue

                loss = (  # protect sub task
                    top_l+ sub_l+sent_l)    # sentiment still important

            top_loss += top_l.item()
            sub_loss += sub_l.item()
            sent_loss += sent_l.item()
            current_step += 1

            if cfg["use_amp"]:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            scheduler.step()

        avg_top_loss = top_loss / len(train_loader)
        avg_sub_loss = sub_loss / len(train_loader)
        avg_sent_loss = sent_loss / len(train_loader)

        print(f"Top Loss: {avg_top_loss:.4f}")
        print(f"sub Loss: {avg_sub_loss:.4f}")
        print(f"sent Loss: {avg_sent_loss:.4f}")

        val_results = evaluate_joint_with_sentiments(
            model, val_loader, DEVICE, epoch,
            top_thresholds=np.array([0.5]*top_dim),
            sub_thresholds=np.array([0.5]*sub_dim,
            use_hier_mask=True, eval_data=eval_data)
        f1 = val_results["sent_f1"]
        best_f1 = 0

        print(f"Val — Top F1: {val_results['top_f1']:.4f}, "
              f"Sub F1: {val_results['sub_f1']:.4f}, "
              f"HierAcc: {val_results['hier_acc']:.4f}")
        
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "/content/best_stage3.pt")

    print("\n Training complete.")
    return model

In [ ]:
CFG = {
    "hf_repo": "Faisal191/aspect-classifier", 
    "hf_encoder_subfolder": "Domain_trained_encoder", # Specify the subfolder separately
    "hierarchical_json": r"/content/final_aspa_data_hierarchical_with_sentiments_temp_v6.json",  # your prepared hierarchical JSON
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "batch_size": 16,
    "epochs": 5,
    "lr_encoder": 5e-6,
    "lr_heads": 1e-5,
    "lr_sent": 2e-5,
    "lr_sent_encoder": 2e-5,
    "use_amp": True,
    "weight_decay": 0.01,
    "max_len": 256,
    "freeze": True,   # True: freeze encoder (only train sentiment head); False: fine-tune encoder too
    "freeze_epochs": 0,        # Number of initial epochs to freeze the encoder
    "use_aspect_embedding": True, # alternative to one-hot: learn small embedding per sub-cluster
    "aspect_embedding_dim": 768,
    "save_dir": "./sentiment_head_ckpt",
    "seed": 42,
    "label_map": { 0: 0, 1: 1},
    "num_classes": 2,
}
Path(CFG["save_dir"]).mkdir(parents=True, exist_ok=True)
torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
DEVICE = torch.device(CFG["device"])


In [ ]:
def evaluate_joint_with_sentiments(model, dataloader, device, epoch,
    top_thresholds=None, sub_thresholds=None, use_hier_mask=True, sent_threshold = 0.35, eval_data=None):

    model.eval()
    all_y_top, all_p_top = [], []
    all_y_sub, all_p_sub = [], []
    all_p_top_cont, all_p_sub_cont = [], []

    all_sent_preds = []
    all_sent_labels = []
    all_example_list = []
    all_sent_sub = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Eval E{epoch}"):
            ids, mask = batch["input_ids"].to(device), batch["attention_mask"].to(device)
            y_top = batch["top_labels"].cpu().numpy()
            y_sub = np.stack([s.to_dense().squeeze(0).cpu().numpy() for s in batch["sub_labels"]])
            sentiments = batch["sentiments"]

            outputs = model(ids, mask, sentiments_dict_batch=sentiments)

            top_logits = outputs["top_logits"].cpu().numpy()
            sub_logits = outputs["sub_logits"].cpu().numpy()
            p_top = torch.sigmoid(torch.tensor(top_logits)).numpy()
            p_sub = torch.sigmoid(torch.tensor(sub_logits)).numpy()
            # --- hierarchical masking ---
            if use_hier_mask and top_thresholds is not None:
                p_sub_masked = apply_hierarchical_mask_to_sub_probs(p_top, p_sub, top_thresholds, model._get_dense_map())
            else:
                p_sub_masked = p_sub

            if top_thresholds is not None:
                p_top_bin = (p_top > np.array(top_thresholds)).astype(int)
            else:
                p_top_bin = (p_top > 0.5).astype(int)

            if sub_thresholds is not None:
                p_sub_bin = (p_sub_masked > np.array(sub_thresholds)[None, :]).astype(int)
            else:
                p_sub_bin = (p_sub_masked > 0.5).astype(int)

            all_y_top.append(y_top)
            all_p_top.append(p_top_bin)
            all_y_sub.append(y_sub)
            all_p_sub.append(p_sub_bin)
            all_p_top_cont.append(p_top)
            all_p_sub_cont.append(p_sub_masked)

            for s_logits_, s_labels, s_example, sub_id, s_logits in zip(outputs["sent_logits_list"], outputs["sent_labels_list"], outputs["sent_example_map"], outputs["sent_sub"], outputs["sent_logits_adap"]):
                if s_labels.numel() == 0:
                    continue
                #s_logits = 0.5*(s_logits + s_logits_)
                #diff = torch.abs(s_logits[:, 1] - s_logits[:, 0])
                #tau = 0.5  # tune on validation
                preds = torch.argmax(s_logits, dim=-1).cpu().numpy()

                """# mark low-confidence predictions
                mask = (diff >= tau).cpu().numpy()

                preds[~mask] = -1"""
                labels = s_labels.cpu().numpy()

                all_example_list.extend([s_example]*len(labels))
                all_sent_preds.extend(preds.tolist())
                all_sent_labels.extend(labels.tolist())
                all_sent_sub.extend(sub_id)

    y_top = np.vstack(all_y_top)
    p_top = np.vstack(all_p_top)
    y_sub = np.vstack(all_y_sub)
    p_sub = np.vstack(all_p_sub)

    # Compute hierarchical F1 metrics
    _, _, f1_top = compute_metrics(y_top, p_top)
    _, _, f1_sub = compute_metrics(y_sub, p_sub)


    hier_acc = np.mean(
        ((y_top * p_top).sum(1) > 0)
        & ((y_sub * p_sub).sum(1) > 0)
    )

    if len(all_sent_labels) > 0:
        print(f"Unique sentiment labels in validation set: {np.unique(all_sent_labels)}")
        sent_prec = precision_score(all_sent_labels, all_sent_preds, average="macro", zero_division=0)
        sent_rec = recall_score(all_sent_labels, all_sent_preds, average="macro", zero_division=0)
        sent_f1 = f1_score(all_sent_labels, all_sent_preds, average="macro", zero_division=0)

        sent_conf_mat = confusion_matrix(all_sent_labels, all_sent_preds)
    else:
        sent_prec = sent_rec = sent_f1 = 0
        sent_conf_mat = np.zeros((2, 2))
    print("Top f1", f1_top)
    print("sub f1", f1_sub)
    print("hier acc", hier_acc)
    print("sentiment_precision", sent_prec)
    print("sentiment_recall", sent_rec)
    print("sentiment_f1", sent_f1)
    print("sentiment_confusion matrix\n", sent_conf_mat)

    for i in range(sub_logits.shape[-1]): # Changed .size(-1) to .shape[-1]
        indx = np.where(np.array(all_sent_sub) == i)[0]
        if len(indx) > 0:
            sub_preds = np.array(all_sent_preds)[indx]
            sub_labels = np.array(all_sent_labels)[indx]
            f1 = f1_score(sub_preds, sub_labels, average="macro", zero_division=0)
            print(f"Sub aspect {i} and f1 score {f1}")
        else:
            print(f"Sub aspect {i} no sentiment for this aspect")

    if epoch >= 15:
      for i in range(len(all_sent_preds)):
        print(eval_data[all_example_list[i]])
        print(f"sub_aspect is {all_sent_sub[i]}, label {all_sent_labels[i]} Predicted {all_sent_preds[i]}")


    return {
        "top_f1": f1_top,
        "sub_f1": f1_sub,
        "hier_acc": hier_acc,
        "sent_precision": sent_prec,
        "sent_recall": sent_rec,
        "sent_f1": sent_f1,
        "sent_confusion_matrix": sent_conf_mat,
    }

In [ ]:
if __name__ == "__main__":
    model = train_sentiment_head(CFG)

In [ ]:
hf_token = userdata.get('HF_TOKEN')

# Initialize HfApi with the token
api = HfApi(token=hf_token)

username = "Faisal191"
repo_name = "aspect-classifier"
repo_id = f"{username}/{repo_name}"

checkpoint_path = Path(r"/content/stage2_joint/best_stage2.pt")

# Attempt the upload again with proper authentication
api.upload_file(
    path_or_fileobj=checkpoint_path,
    path_in_repo="final_stage3_v6/best_stage3.pt",
    repo_id=repo_id,
    commit_message="Upload final checkpoint")